# Valeurs foncières ingestion

Use the helper functions in `valeurs_historique.py` to download DVF datasets, optionally geocode the rows via the local Nominatim instance, and export the result.

In [ ]:
import pandas as pd

from valeurs_historique import (
    data_url_2025,
    geocode_dataframe,
    load_valeurs_foncieres_dataframe,
    save_geodataframe,
)

print(data_url_2025)

## Download the raw DVF data

In [ ]:
data_urls = [data_url_2025]
separator = "|"

df = load_valeurs_foncieres_dataframe(data_urls, sep=separator)
print(f"Loaded {len(df):,} rows from {len(data_urls)} file(s).")

# Keep only rows where `Type local` is 'Maison' or 'Appartement'
df = df[df["Type local"].isin(["Maison", "Appartement"])]
print(f"Filtered to {len(df):,} rows for 'Maison' and 'Appartement'.")

# Cast columns to appropriate types
df["Valeur fonciere"] = df["Valeur fonciere"].str.replace(",", ".").astype(float)
df["Surface reelle bati"] = df["Surface reelle bati"].str.replace(",", ".").astype(float)
df["Surface terrain"] = df["Surface terrain"].str.replace(",", ".").astype(float)
df["Date mutation"] = pd.to_datetime(df["Date mutation"], format="%d/%m/%Y")

# Add column for price to surface ratio
df["price_per_sqm"] = df["Valeur fonciere"] / df["Surface reelle bati"]
# Add column for price to terrain surface ratio
df["price_per_sqm_terrain"] = df["Valeur fonciere"] / df["Surface terrain"]

df.head()

## Geocode rows with the local Nominatim service

In [ ]:
# # Keep only n rows for testing
# df = df.sample(n=1000, random_state=42)

In [ ]:
geo_df, geocode_errors = geocode_dataframe(df)

# Print ratio of empty geometries
empty_geom_ratio = geo_df["geometry"].isna().mean()
print(f"Ratio of empty geometries: {empty_geom_ratio:.2%}")
print(f"Number of empty geometries: {geo_df['geometry'].isna().sum()} out of {len(geo_df)}")
print(f"Number of geocoding errors: {len(geocode_errors)}")
for i in geocode_errors:
    print(i)

In [ ]:
geo_df.head()

## Export as GeoJSON or GeoPackage

In [ ]:
save_geodataframe(geo_df, "valeurs_foncieres_2025.gpkg", layer="valeurs_foncieres")
# save_geodataframe(geo_df, "valeurs_foncieres_2025.parquet")